In [1]:
%pip install webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas


Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [1]:
# imports clave para el funcionamiento de todo el scrapping

from selenium import webdriver
from selenium. webdriver.chrome.options import Options
from selenium. webdriver.common.by import By
import pandas as pd
import time
import re

In [2]:
#imports necesarios para pasar página y seguir escrapeando más de 10 artículos

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
#Agregado para probar una mejor versión del scrapping

import requests
import json

In [4]:
# ==========================
# CONFIGURACIÓN
# ==========================
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-notifications")

driver = webdriver.Chrome(options=options)

url = "https://www.elespectador.com/buscador/migraci%C3%B3n-venezolana/"
driver.get(url)

print("⏳ Esperando carga inicial...")
time.sleep(10)

⏳ Esperando carga inicial...


In [8]:
# ==========================
# SCROLL MODERADO
# ==========================
print("\n📜 Haciendo scroll...")

for i in range(10):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(4)
    print(f"   Scroll {i+1}/10")

print("\n" + "="*60)
print("🔍 INICIANDO EXTRACCIÓN CON DEBUGGING")
print("="*60)

# up to this point everthing is good.


📜 Haciendo scroll...
   Scroll 1/10
   Scroll 2/10
   Scroll 3/10
   Scroll 4/10
   Scroll 5/10
   Scroll 6/10
   Scroll 7/10
   Scroll 8/10
   Scroll 9/10
   Scroll 10/10

🔍 INICIANDO EXTRACCIÓN CON DEBUGGING


In [9]:
# ==========================
# DEBUGGING PASO A PASO
# ==========================

# PASO 1: Ver todos los H2 y H3
print("\n📌 PASO 1: Buscando todos los H2 y H3...")
h2_todos = driver.find_elements(By.TAG_NAME, 'h2')
h3_todos = driver.find_elements(By.TAG_NAME, 'h3')
print(f"   ✓ H2 encontrados: {len(h2_todos)}")
print(f"   ✓ H3 encontrados: {len(h3_todos)}")

# PASO 2: Ver H2/H3 con enlaces dentro
print("\n📌 PASO 2: Buscando H2 y H3 con enlaces <a>...")
h2_con_enlaces = driver.find_elements(By.XPATH, '//h2/a')
h3_con_enlaces = driver.find_elements(By.XPATH, '//h3/a')
print(f"   ✓ H2 con <a>: {len(h2_con_enlaces)}")
print(f"   ✓ H3 con <a>: {len(h3_con_enlaces)}")

# Mostrar ejemplos
if h2_con_enlaces: 
    print(f"\n   Ejemplo H2:")
    for i, elem in enumerate(h2_con_enlaces[: 3]):
        print(f"      {i+1}.  Texto: '{elem.text[: 60]}...'")
        print(f"         URL: {elem.get_attribute('href')}")

if h3_con_enlaces:
    print(f"\n   Ejemplo H3:")
    for i, elem in enumerate(h3_con_enlaces[:3]):
        print(f"      {i+1}.  Texto: '{elem.text[: 60]}...'")
        print(f"         URL: {elem.get_attribute('href')}")

# PASO 3: Buscar por IDs específicos
print("\n📌 PASO 3: Buscando en bloques específicos...")
bloques_ids = [
    'main-layout-2',
    'main-layout-6-7',
    'home_bloque_Reportajes'
]

elementos_por_bloque = []
for bloque_id in bloques_ids: 
    try:
        # Buscar el bloque
        bloque = driver.find_elements(By.ID, bloque_id)
        if bloque:
            # Buscar enlaces dentro del bloque
            enlaces_h2 = driver.find_elements(By. XPATH, f'//*[@id="{bloque_id}"]//h2/a')
            enlaces_h3 = driver.find_elements(By.XPATH, f'//*[@id="{bloque_id}"]//h3/a')
            total = len(enlaces_h2) + len(enlaces_h3)
            print(f"   ✓ {bloque_id}: {total} enlaces (H2: {len(enlaces_h2)}, H3: {len(enlaces_h3)})")
            elementos_por_bloque.extend(enlaces_h2)
            elementos_por_bloque.extend(enlaces_h3)
        else:
            print(f"   ✗ {bloque_id}: NO ENCONTRADO")
    except Exception as e:
        print(f"   ✗ {bloque_id}: ERROR - {e}")

# PASO 4: Buscar TODOS los enlaces que contengan "politica"
print("\n📌 PASO 4: Todos los enlaces que contengan 'politica'...")
todos_enlaces_politica = driver.find_elements(By. XPATH, '//a[contains(@href, "/politica/")]')
print(f"   ✓ Enlaces con '/politica/': {len(todos_enlaces_politica)}")

# Mostrar algunos ejemplos
if todos_enlaces_politica:
    print(f"\n   Primeros 5 enlaces con '/politica/':")
    for i, elem in enumerate(todos_enlaces_politica[:5]):
        texto = elem.text. strip()
        url = elem.get_attribute('href')
        print(f"      {i+1}. '{texto[: 50]}...' -> {url}")


📌 PASO 1: Buscando todos los H2 y H3...
   ✓ H2 encontrados: 10
   ✓ H3 encontrados: 21

📌 PASO 2: Buscando H2 y H3 con enlaces <a>...
   ✓ H2 con <a>: 10
   ✓ H3 con <a>: 10

   Ejemplo H2:
      1.  Texto: 'Migrantes en Bello: así deben actualizar sus datos para no p...'
         URL: https://www.elespectador.com/mundo/venezuela/migrantes-en-bello-antioquia-asi-deben-actualizar-sus-datos-para-no-perder-el-servicio-de-salud/
      2.  Texto: 'Petro en EE. UU.: “Hay que avanzar más” en los derechos de l...'
         URL: https://www.elespectador.com/mundo/america/petro-en-ee-uu-hay-que-avanzar-mas-en-los-derechos-de-los-migrantes-venezolanos/
      3.  Texto: '“Emigro, luego existo”: la diáspora venezolana narrada desde...'
         URL: https://www.elespectador.com/el-magazin-cultural/emigro-luego-existo-la-diaspora-venezolana-narrada-desde-el-grafiti/

   Ejemplo H3:
      1.  Texto: 'Redacción Internacional...'
         URL: None
      2.  Texto: 'Redacción Internacional...'
      

In [ ]:
# ============================================
# FUNCIÓN PARA LA EXTRACCIÓN DE LOS ARTÍCULOS
# ============================================


def extraction():

    print("\n" + "=" * 60)
    print("📊 EXTRAYENDO DATOS DESDE API searcherTag")
    print("=" * 60)

    titulares = []
    links = []
    categorias = []
    cuerpos = []

    api_url = "https://www.elespectador.com/pf/api/v3/content/fetch/searcherTag"

    # ==========================
    # OBTENER TODAS LAS NOTICIAS
    # ==========================

    total_noticias = None

    for offset in range(0, 20, 10):

        print(f"\n📄 Descargando bloque desde {offset}")

        query = {
            "author": None,
            "date": None,
            "from": offset,
            "keyword": "migración-venezolana",
            "section": None,
            "size": 10,
            "subtype": None
        }

        params = {
            "query": json.dumps(query, ensure_ascii=False),
            "d": "1198",
            "mxId": "00000000",
            "_website": "el-espectador"
        }

        try:

            response = requests.get(
                api_url,
                params=params,
                timeout=30
            )

            response.raise_for_status()

            data = response.json()

            # Obtener total de resultados sólo una vez
            if total_noticias is None:
                total_noticias = data.get("count", 0)

                print(
                    f"✅ Total de noticias encontradas: "
                    f"{total_noticias}"
                )

            noticias = data.get("content_elements", [])

            if len(noticias) == 0:
                print("⚠️ No se encontraron más noticias")
                break

            for noticia in noticias:

                try:

                    titulo = noticia["headlines"]["basic"]

                    categoria = noticia["taxonomy"][
                        "primary_section"
                    ]["name"]

                    url = (
                        "https://www.elespectador.com"
                        + noticia["canonical_url"]
                    )

                    titulares.append(titulo)
                    categorias.append(categoria)
                    links.append(url)

                except Exception as e:
                    print(f"⚠️ Error procesando noticia: {e}")

            # Si ya llegamos al total, terminar
            if total_noticias and len(titulares) >= total_noticias:
                break

        except Exception as e:
            print(f"❌ Error API: {e}")
            break

    print("\n" + "=" * 60)
    print("✅ EXTRACCIÓN DE METADATOS COMPLETADA")
    print("=" * 60)

    print(f"Títulos encontrados: {len(titulares)}")
    print(f"Links encontrados: {len(links)}")

    # ==========================
    # EXTRAER CUERPO ARTÍCULOS
    # ==========================

    print("\n" + "=" * 60)
    print("📰 EXTRAYENDO CUERPOS DE ARTÍCULOS")
    print("=" * 60)

    for i, url in enumerate(links, start=1):

        try:

            driver.get(url)

            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located(
                    (By.TAG_NAME, "article")
                )
            )

            parrafos = driver.find_elements(
                By.CSS_SELECTOR,
                "article p"
            )

            if not parrafos:

                parrafos = driver.find_elements(
                    By.CSS_SELECTOR,
                    '[class*="article"] p'
                )

            if not parrafos:

                parrafos = driver.find_elements(
                    By.TAG_NAME,
                    "p"
                )

            texto_articulo = " ".join(

                p.text.strip()

                for p in parrafos

                if len(p.text.strip()) > 30

            )

            cuerpos.append(texto_articulo)

        except Exception as e:

            print(f"❌ Error en {url}: {e}")

            cuerpos.append("")

    

In [ ]:
def save_extraction():
        # ==========================
        # DATAFRAME FINAL
        # ==========================

    df = pd.DataFrame({

        "Titulo": titulares,

        "Categoria": categorias,

        "URL": links,

            "Cuerpo": cuerpos

    })

    print("\n" + "=" * 60)
    print("📈 RESUMEN FINAL")
    print("=" * 60)

    print(f"Noticias extraídas: {len(df)}")

    df.to_csv(
        "migracion_venezolana.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("✅ Archivo guardado: migracion_venezolana.csv")

    return df

In [ ]:
extraction()
save_extraction()
